# Setting up UniCAS slide-level inference
### credits: https://github.com/peter-fei/UniCAS

## Modules

In [1]:
import json
import os
import time
import uuid
from types import SimpleNamespace
from collections import OrderedDict
from functools import partial
from pathlib import Path
from typing import Callable, Dict, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.io import decode_image
from torchvision.transforms import v2

## multitask_agg.py

In [2]:
class PPEG(nn.Module):
    """Depth-wise convolutional positional encoding."""

    def __init__(self, dim=512):
        super().__init__()
        self.proj = nn.Conv2d(dim, dim, 7, 1, 7 // 2, groups=dim)
        self.proj1 = nn.Conv2d(dim, dim, 5, 1, 5 // 2, groups=dim)
        self.proj2 = nn.Conv2d(dim, dim, 3, 1, 3 // 2, groups=dim)

    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        B, _, C = x.shape
        cnn_feat = x.transpose(1, 2).view(B, C, H, W)
        x = self.proj(cnn_feat) + cnn_feat + self.proj1(cnn_feat) + self.proj2(cnn_feat)
        x = x.flatten(2).transpose(1, 2)
        return x

In [3]:
class TaskAttention(nn.Module):
    """Task-specific attention that routes each task token to top-k patch tokens."""

    def __init__(self,
                 dim,   # 输入token的dim
                 num_heads=8,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop_ratio=0.,
                 proj_drop_ratio=0.,
                 num_tasks=1, capacity_factor=2):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qs = nn.ModuleList([nn.Linear(dim, dim, bias=qkv_bias) for _ in range(num_tasks)])
        self.kv = nn.Linear(dim, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop_ratio)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop_ratio)
        self.num_tasks = num_tasks
        self.capacity_factor = capacity_factor

        self.in_features = dim
        self.experts = nn.ModuleList([nn.Linear(dim, dim, bias=False) for _ in range(self.num_tasks)])

    def forward(self, x: torch.Tensor, heads=None) -> torch.Tensor:
        B, N, C = x.shape  # x: [B, num_tasks + tokens, C]
        feature = x[:, self.num_tasks :, :]  # feature: [B, tokens, C]
        feature_multihead = feature.reshape(B, -1, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)  # [B, H, tokens, Dh]
        kv = self.kv(feature).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)  # [2, B, H, tokens, Dh]
        k, v = kv[0], kv[1]  # k,v: [B, H, tokens, Dh]

        # Limit attention to the top-k tokens per task to reduce compute.
        top_k = max(1, min(self.capacity_factor, feature.size(1)))

        qs = []  # will become [B, H, num_tasks, 1, Dh]
        for i in range(self.num_tasks):
            tasktoken = x[:, i, :].unsqueeze(1)
            q = self.qs[i](tasktoken).reshape(B, -1, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
            qs.append(q)

        qs = torch.cat(qs, dim=2)  # [B, H, num_tasks, 1, Dh]

        atten_tf = (qs @ k.transpose(-2, -1)) * self.scale  # [B, H, num_tasks, tokens]
        atten_tf = self.attn_drop(atten_tf)

        top_k_value, top_k_indice = torch.topk(atten_tf, k=top_k, dim=-1, sorted=False)  # both [B, H, num_tasks, top_k]
        top_k_value_softmax = top_k_value.softmax(dim=-1)

        atten_topk = torch.zeros_like(atten_tf)  # [B, H, num_tasks, tokens]
        atten_topk.scatter_(-1, top_k_indice, top_k_value_softmax)
        self.last_attn = atten_topk.detach()     # [NEW]

        attn_token = (atten_topk @ v).transpose(1, 2).reshape(B, -1, C)  # [B, num_tasks, C]

        feature_multihead_expand = feature_multihead.unsqueeze(2).expand(-1, -1, self.num_tasks, -1, -1)  # [B, H, num_tasks, tokens, Dh]
        top_k_indice_expand = top_k_indice.unsqueeze(-1).expand(-1, -1, -1, -1, v.size(-1))  # [B, H, num_tasks, top_k, Dh]
        feature_topk = torch.gather(feature_multihead_expand, -2, top_k_indice_expand)  # [B, H, num_tasks, top_k, Dh]

        feature_topk *= top_k_value_softmax.unsqueeze(-1).expand_as(feature_topk)

        feature_pad = torch.zeros_like(feature_multihead_expand)  # [B, H, num_tasks, tokens, Dh]
        feature_pad.scatter_(-2, top_k_indice_expand, feature_topk)
        feature_pad = feature_pad.permute(0, 2, 3, 1, 4).reshape(B, self.num_tasks, -1, C)  # [B, num_tasks, tokens, C]

        feature_output = torch.stack(
            [f_i(feature_pad[:, i, :, :]) for i, f_i in enumerate(self.experts)], dim=1
        )
        token_output = torch.stack(
            [f_i(attn_token[:, i, :]) for i, f_i in enumerate(self.experts)], dim=1
        )
        task_topk_weight = torch.ones_like(feature_output)

        feature_output = torch.einsum("btnc,btnc->bnc", feature_output, task_topk_weight)
        output = torch.cat((token_output, feature_output), dim=1)
        return output

In [4]:
def drop_path(x: torch.Tensor, drop_prob: float = 0.0, training: bool = False) -> torch.Tensor:
    """Stochastic depth per sample when applied to residual paths."""

    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    output = x.div(keep_prob) * random_tensor
    return output

class DropPath(nn.Module):
    """Drop paths (stochastic depth) per sample for residual blocks."""

    def __init__(self, drop_prob: Optional[float] = None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return drop_path(x, self.drop_prob, self.training)

In [5]:
def softmax(x: torch.Tensor, dim: Union[int, tuple[int, ...]]) -> torch.Tensor:
    """
    Compute the softmax along the specified dimensions.
    This function adds the option to specify multiple dimensions

    Args:
        x (torch.Tensor): Input tensor.
        dims (int or tuple[int]): The dimension or list of dimensions along which the softmax probabilities are computed.

    Returns:
        torch.Tensor: Output tensor containing softmax probabilities along the specified dimensions.
    """
    max_vals = torch.amax(x, dim=dim, keepdim=True)
    e_x = torch.exp(x - max_vals)
    sum_exp = e_x.sum(dim=dim, keepdim=True)
    return e_x / sum_exp


class SoftMoELayerWrapper(nn.Module):
    """
    A wrapper class to create a Soft Mixture of Experts layer.

    From "From Sparse to Soft Mixtures of Experts"
    https://arxiv.org/pdf/2308.00951.pdf
    """

    def __init__(
        self,
        dim: int,
        num_experts: int,
        slots_per_expert: int,
        layer: Callable,
        normalize: bool = True,
        **layer_kwargs,
    ) -> None:
        """
        Args:
            dim (int): Dimensionality of input features.
            num_experts (int): Number of experts.
            slots_per_expert (int): Number of token slots per expert.
            layer (Callable): Network layer of the experts.
            normalize (bool): Normalize input and phi (sec. 2.3 from paper)
            **layer_kwargs: Additional keyword arguments for the layer class.
        """
        super().__init__()
        print('SoftMoELayerWrapper',dim,num_experts,slots_per_expert)
        self.dim = dim
        self.num_experts = num_experts
        self.slots_per_expert = slots_per_expert
        self.normalize = normalize

        # Initialize phi and normalization scaling factor
        self.phi = nn.Parameter(torch.zeros(dim, num_experts, slots_per_expert))
        if self.normalize:
            self.scale = nn.Parameter(torch.ones(1))

        # Initialize phi using LeCun normal initialization
        # https://github.com/google-research/vmoe/blob/662341d007650d5bbb7c6a2bef7f3c759a20cc7e/vmoe/projects/soft_moe/router.py#L49C1-L49C1
        nn.init.normal_(self.phi, mean=0, std=1 / dim**0.5)

        # Create a list of expert networks
        self.experts = nn.ModuleList(
            [layer(**layer_kwargs) for _ in range(num_experts)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the Soft-MoE layer (algorithm 1 from paper).

        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, input_dim].

        Returns:
            torch.Tensor: Output tensor of shape [batch_size, seq_len, input_dim].
        """
        assert (
            x.shape[-1] == self.dim
        ), f"Input feature dim of {x.shape[-1]} does not match layer dim of {self.dim}"
        assert (
            len(x.shape) == 3
        ), f"Input expected to have 3 dimensions but has {len(x.shape)}"

        phi = self.phi

        # Normalize input and phi
        if self.normalize:
            x = F.normalize(x, dim=2)  # [b, m, d]
            phi = self.scale * F.normalize(phi, dim=0)  # [d, n, p]

        # Compute dispatch and combine weights
        logits = torch.einsum("bmd,dnp->bmnp", x, phi)
        d = softmax(logits, dim=1)
        c = softmax(logits, dim=(2, 3))

        # Compute input slots as weighted average of input tokens using dispatch weights
        xs = torch.einsum("bmd,bmnp->bnpd", x, d)

        # Apply expert to corresponding slots
        ys = torch.stack(
            [f_i(xs[:, i, :, :]) for i, f_i in enumerate(self.experts)], dim=1
        )

        # Compute output tokens as weighted average of output slots using combine weights
        y = torch.einsum("bnpd,bmnp->bmd", ys, c)

        return y

In [6]:
class Mlp(nn.Module):
    """MLP as used in ViT / MLP-Mixer."""

    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0, bias=True):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features, bias=bias)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features, bias=bias)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

In [7]:
class ExpChoiceBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False, qk_scale=None, drop=0.0, attn_drop=0.0,
                 drop_path_ratio=0.0, act_layer=nn.GELU, norm_layer=nn.LayerNorm, num_experts=8, num_tasks=3,
                 capacity_factor=256):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.norm2 = norm_layer(dim)
        self.attn = TaskAttention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop_ratio=attn_drop,
            proj_drop_ratio=drop,
            num_tasks=num_tasks,
            capacity_factor=capacity_factor,
        )
        self.drop_path = DropPath(drop_path_ratio) if drop_path_ratio > 0.0 else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.num_tasks = num_tasks

        # SoftMoE routes tokens to experts in the feed-forward block.
        self.mlp = SoftMoELayerWrapper(
            dim,
            num_experts=num_experts,
            slots_per_expert=1,
            layer=partial(Mlp, in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop),
            normalize=True,
        )

        self.mlp_drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor, heads=None) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x), heads))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [8]:
def _init_vit_weights(m):
    """
    ViT weight initialization
    :param m: module
    """
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)

class MultiTask_Agg(nn.Module):
    """Multi-task aggregation transformer with task tokens and MoE MLPs."""

    def __init__(self, patch_size=16, in_c=3, classes=None,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, qkv_bias=True,
                 qk_scale=None, representation_size=None, distilled=False, drop_ratio=0.0,
                 attn_drop_ratio=0.0, drop_path_ratio=0.0, norm_layer=None,
                 act_layer=None, num_tasks=1, num_experts=16, moe_gate_dim=770, capacity_factors=None):

        super().__init__()
        if classes is None:
            classes = [2, 2, 2]
        if isinstance(classes, int):
            classes = [classes] * num_tasks

        self.classes = classes
        embed_dim_in = embed_dim
        self.embed_dim = 768  # embed_dim for consistency with other models
        self.num_tokens = 2 if distilled else 1
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        act_layer = act_layer or nn.GELU

        self.dist_token = nn.Parameter(torch.zeros(1, 1, self.embed_dim)) if distilled else None
        self.pos_drop = nn.Dropout(p=drop_ratio)

        self.depth = depth
        self.norm_layer = norm_layer
        self.num_experts = num_experts

        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.drop_rate = drop_ratio
        self.attn_drop_rate = attn_drop_ratio
        self.num_tasks = num_tasks
        self.moe_gate_dim = moe_gate_dim

        blocks = []
        capacity_factors = capacity_factors or [256] * depth
        if len(capacity_factors) != depth:
            raise ValueError(f"capacity_factors length ({len(capacity_factors)}) must equal depth ({depth})")

        for i in range(self.depth):
            blocks.append(
                ExpChoiceBlock(
                    dim=self.embed_dim,
                    num_heads=self.num_heads,
                    mlp_ratio=self.mlp_ratio,
                    qkv_bias=self.qkv_bias,
                    qk_scale=self.qk_scale,
                    num_experts=num_experts,
                    drop=self.drop_rate,
                    attn_drop=self.attn_drop_rate,
                    drop_path_ratio=drop_path_ratio,
                    norm_layer=self.norm_layer,
                    num_tasks=num_tasks,
                    capacity_factor=capacity_factors[i],
                )
            )

        self.blocks = nn.Sequential(*blocks)

        self.layer1 = nn.Sequential(
            nn.Linear(embed_dim_in, self.embed_dim),
            nn.LayerNorm(self.embed_dim),
            nn.GELU(),
        )

        self.norm = norm_layer(self.embed_dim)

        # Representation layer
        if representation_size and not distilled:
            self.has_logits = True
            self.embed_dim = representation_size
            self.pre_logits = nn.Sequential(OrderedDict([
                ("fc", nn.Linear(self.embed_dim, representation_size)),
                ("act", nn.Tanh())
            ]))
        else:
            self.has_logits = False
            self.pre_logits = nn.Identity()

        self.head_dist = None
        if distilled:
            self.head_dist = nn.Linear(self.embed_dim, self.classes[1]) if classes > 0 else nn.Identity()

        self.task_tokens = nn.Parameter(torch.zeros(1, self.num_tasks, self.embed_dim))

        self.heads = nn.ModuleList([nn.Linear(self.embed_dim, classes[i]) for i in range(self.num_tasks)])

        self.pos_layer = PPEG(dim=self.embed_dim)

        # Weight init
        if self.dist_token is not None:
            nn.init.trunc_normal_(self.dist_token, std=0.02)

        nn.init.trunc_normal_(self.task_tokens, std=0.02)
        self.apply(_init_vit_weights)


    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(x)

        H = x.shape[1]
        _H, _W = int(np.ceil(np.sqrt(H))), int(np.ceil(np.sqrt(H)))
        add_length = _H * _W - H
        if add_length > 0:
            # Pad to a square sequence so the depth-wise convolutions see a grid.
            x = torch.cat([x, x[:, :add_length, :]], dim=1)
        x = self.pos_layer(x, _H, _W)

        task_tokens = repeat(self.task_tokens, '() n d -> b n d', b=x.size(0))
        if self.dist_token is None:
            x = torch.cat((task_tokens, x), dim=1)
        else:
            x = torch.cat((task_tokens, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1)

        for blk in self.blocks:
            x = blk(x)

        return x


    def forward(self, x: torch.Tensor) -> Dict[int, torch.Tensor]:
        x = self.forward_features(x)
        x = self.norm(x)
        res: Dict[int, torch.Tensor] = {}
        for i in range(self.num_tasks):
            res[int(i)] = self.heads[i](x[:, i])
        return res

## slide_multitask.py

In [9]:
class Slide_Multitask(nn.Module):
    def __init__(self,args=None):
        super().__init__()
        self.img_size = 224
        aggregator = getattr(args,'aggregator','multi_task')
        embed_dim = getattr(args,'embed_dim',1024)
        classes = getattr(args,'classes',[2,2,2])
        num_tasks = getattr(args,'num_tasks',3)
        depth = getattr(args,'depth',8)
        gate_dim = embed_dim
        num_experts = getattr(args,'num_experts',8)
        capacity_factors =  getattr(args,'capacity_factors',256)

        print(f' encoder is {args.encoder}, embed_dim is {embed_dim}----------------------------------------------------------------')
        if aggregator == 'multi_task':
            self.model = MultiTask_Agg(embed_dim=embed_dim, classes=classes, num_tasks=num_tasks,depth=depth, num_experts=num_experts,moe_gate_dim=gate_dim, capacity_factors=capacity_factors)

    def forward(self, x):
        assert len(x.shape) == 3, 'Need to Feature Encoding First !'
        preds = self.model(x)
        return preds

## Loading model

In [10]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/Lucas"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
weights  = base_dir + "/multi-task.pth"
device = "cpu"

args = SimpleNamespace(aggregator="multi_task", embed_dim=1024, classes=[2, 2, 2], num_tasks=3,
                       depth=8, capacity_factors=[256]*8, encoder="UniCAS")
model = Slide_Multitask(args)
print(model.load_state_dict(torch.load(weights, map_location="cpu"), strict=False))
model = model.eval().to(device)

 encoder is UniCAS, embed_dim is 1024----------------------------------------------------------------
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
SoftMoELayerWrapper 768 8 1
<All keys matched successfully>


# Attention (How to get top-k tiles for cancer?)

In [ ]:
torch.manual_seed(0)
feats = torch.randn(1, 5, 1024)

with torch.no_grad():
    preds = model(feats.to(device))

for i, name in enumerate(["cancer", "candidiasis", "clue cell"]):
    logits = preds[i]
    print(f"{name:<15} {logits}")

cancer          tensor([[-3.4017,  3.0268]])
candidiasis     tensor([[ 2.2759, -2.2638]])
clue cell       tensor([[ 3.5995, -2.8210]])


In [ ]:
# 12 heads atenção
# Cada head 3 vetores (cancer/candidiasis/clue cell)
# Cada vetor de tamanho quadrado perfeito mais próximo ao tamanho do input
# Ex.: torch.randn(1,5,1024) -> 9 = 3² mais perto de 5
model.model.blocks[-1].attn.last_attn.shape

torch.Size([1, 12, 3, 9])

In [ ]:
print(model.model.blocks[-1].attn.last_attn)

tensor([[[[1.7956e-01, 5.9020e-02, 2.1530e-01, 5.4162e-03, 7.1228e-02,
           1.7742e-01, 6.2137e-02, 2.2492e-01, 4.9961e-03],
          [4.4595e-02, 1.9300e-02, 4.3185e-01, 2.3359e-04, 1.0821e-03,
           5.1574e-02, 1.7832e-02, 4.3334e-01, 1.9609e-04],
          [1.6326e-01, 1.2929e-01, 1.0245e-01, 2.8409e-02, 1.4218e-01,
           1.6314e-01, 1.3363e-01, 1.1097e-01, 2.6680e-02]],

         [[7.5549e-02, 1.2779e-01, 1.1289e-01, 1.4149e-01, 8.1859e-02,
           7.4771e-02, 1.2637e-01, 1.1478e-01, 1.4450e-01],
          [7.4672e-02, 1.5035e-01, 1.6782e-01, 9.7612e-02, 1.1680e-02,
           8.0404e-02, 1.4177e-01, 1.7824e-01, 9.7457e-02],
          [1.0387e-01, 1.3128e-01, 1.2782e-01, 1.1724e-01, 3.3289e-02,
           1.0796e-01, 1.2741e-01, 1.3431e-01, 1.1682e-01]],

         [[9.0623e-02, 1.2349e-01, 7.0786e-02, 1.3857e-01, 1.5183e-01,
           8.6462e-02, 1.2352e-01, 7.3110e-02, 1.4161e-01],
          [1.5962e-01, 8.3446e-04, 3.0392e-01, 8.7345e-03, 7.6364e-05,
        

In [ ]:
# Apenas os primeiros N scores (N := tamanho do input) importam
# Ex.: torch.randn(1,5,1024), cada vetor de um head de atenção tem tamanho 9
# mas os scores extras se repetem (muito próximos dos primeiros N), seguindo
# o padrão, nesse caso, 0/1/2/3/4/0/1/2/3
for i in range(4):
  print(i,model.model.blocks[-1].attn.last_attn[0][0][0][i])
  print(i+5,model.model.blocks[-1].attn.last_attn[0][0][0][i+5])
  print()

print(4,model.model.blocks[-1].attn.last_attn[0][0][0][8])

0 tensor(0.1796)
5 tensor(0.1774)

1 tensor(0.0590)
6 tensor(0.0621)

2 tensor(0.2153)
7 tensor(0.2249)

3 tensor(0.0054)
8 tensor(0.0050)

4 tensor(0.0050)


In [ ]:
# Vamos usar a média para identificar os top-k tiles para câncer
att_mean_cancer = torch.zeros(5)
for i in range(12):
  # i := i-ésimo attention head // 0 (depois de i) := câncer
  att_mean_cancer += model.model.blocks[-1].attn.last_attn[0][i][0][:5]
print(att_mean_cancer)

# top 3 tiles
_, idx = torch.topk(att_mean_cancer,3)
idx = idx.flatten()
print(idx)

tensor([1.1172, 1.2207, 1.5507, 1.3688, 1.4189])
tensor([2, 4, 3])


# Sanity check on UniCAS's own examples
##### (https://github.com/peter-fei/UniCAS/tree/main/slide-level/Pathology_UniCAS_patch)

## Loading embeddings

In [13]:
C23101400165C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23101400165C10101.pt")
C23063000579C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23063000579C10101.pt")
C23093000068C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23093000068C10101.pt")
C23101500506C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23101500506C10101.pt")
C23090500128C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23090500128C10101.pt")
C23102100075C10101 = torch.load(base_dir + "/UniCAS examples/" + "C23102100075C10101.pt")

unicas_test = [C23101400165C10101, C23063000579C10101, C23093000068C10101, C23101500506C10101, C23090500128C10101, C23102100075C10101]

## Inference

In [28]:
for i, code in enumerate(["C23101400165C10101", "C23063000579C10101", "C23093000068C10101", "C23101500506C10101", "C23090500128C10101", "C23102100075C10101"]):
  print(code)
  with torch.no_grad():
      preds = model(unicas_test[i].unsqueeze(0).to(device))

  for j, name in enumerate(["cancer", "candidiasis", "clue cell"]):
      logits = preds[j]
      print(f"{name:<15} {logits} {logits.argmax().item()}")
  print()

C23101400165C10101
cancer          tensor([[ 1.0486, -1.0359]]) 0
candidiasis     tensor([[-1.0430,  0.7122]]) 1
clue cell       tensor([[ 1.0476, -1.4609]]) 0

C23063000579C10101
cancer          tensor([[-2.7864,  2.4742]]) 1
candidiasis     tensor([[ 0.2226, -0.6591]]) 0
clue cell       tensor([[-2.2329,  2.2209]]) 1

C23093000068C10101
cancer          tensor([[ 0.4280, -0.5016]]) 0
candidiasis     tensor([[ 0.7112, -1.1258]]) 0
clue cell       tensor([[-1.9006,  1.8903]]) 1

C23101500506C10101
cancer          tensor([[ 1.1451, -1.1419]]) 0
candidiasis     tensor([[-0.9446,  0.5786]]) 1
clue cell       tensor([[ 2.0546, -2.3338]]) 0

C23090500128C10101
cancer          tensor([[-0.4916,  0.2993]]) 1
candidiasis     tensor([[ 1.1446, -1.7327]]) 0
clue cell       tensor([[-2.5723,  2.4367]]) 1

C23102100075C10101
cancer          tensor([[ 1.1527, -1.1427]]) 0
candidiasis     tensor([[ 1.3227, -1.5894]]) 0
clue cell       tensor([[ 3.2948, -3.0265]]) 0



## Answer key

In [29]:
pd.read_csv(base_dir + "/UniCAS examples/" + "test.csv").drop(["name","cancer_grade","task_id"], axis=1)

,code,cancer,candidiasis,cluecell
0,C23101400165C10101,0,1,0
1,C23063000579C10101,1,0,1
2,C23093000068C10101,0,0,1
3,C23101500506C10101,0,1,0
4,C23090500128C10101,1,0,1
5,C23102100075C10101,0,0,0


# UniCAS inference on in-house slides

## Choose a slide

In [30]:
slide_name  = "2024-10-14_09_45_11CYT"
drive_slide = base_dir + "/Lâminas Erviegas/" + slide_name + ".kfb"
local_slide = "/content/" + slide_name + ".kfb"

## Sanity check (all tiles loaded)

In [ ]:
!apt-get update -qq
!apt-get install -y openslide-tools libopenslide0

In [ ]:
!pip install openslide-python
!pip install kfbslide

In [33]:
import openslide, kfbslide
import shutil
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv

In [34]:
def count_tiles(slide_path,tile_size=224):
    # Abertura da lâmina e verificação de magnificação 20x
    s = kfbslide.OpenSlide(Path(slide_path))
    if int(s.properties.get("openslide.objective-power")) != 40:
        s.close()
        return
    if int(s.level_downsamples[1]) != 2:
        s.close()
        return

    # Otsu na lâmina toda (miniatura)
    lv_thumb = s.get_best_level_for_downsample(32)
    thumb = s.read_region((0, 0), lv_thumb, s.level_dimensions[lv_thumb]).convert("L")
    thr = threshold_otsu(np.asarray(thumb))
    total_area = tile_size * tile_size

    # Extração de tiles
    level = 1
    step = tile_size * 2
    W, H = s.dimensions
    count = 0
    for y in range(0, H-step+1, step):
        for x in range(0, W-step+1, step):

            region = s.read_region((x, y), level, (tile_size, tile_size))
            if np.asarray(region)[..., 3].min() == 0:
                continue
            img = region.convert("RGB")

            gray = np.asarray(img.convert("L"))
            mask = gray < thr

            cell_area = mask.sum()
            if (cell_area / total_area) < 0.20:
              continue

            # Evita tiles 'sujos'
            px = np.asarray(img)[mask].reshape(-1, 1, 3)
            if np.median(rgb2hsv(px / 255.0)[..., 0]) * 360 < 180:
              continue

            count += 1

    s.close()
    return count

In [35]:
shutil.copy(drive_slide, local_slide)
num_count_tiles = count_tiles(local_slide)
os.remove(local_slide)

In [36]:
torch_embeddings = torch.load(base_dir + "/Embeddings/" + "embeddings_" + slide_name + ".pt")

print(f"Total de tiles (count_tiles): {num_count_tiles}")
print(f"Total de tiles (torch_embed): {torch_embeddings.shape[0]}")

if torch_embeddings.shape[0] != num_count_tiles:
  raise Exception(f"Total de tiles do ambiente de execução difere do drive")
else:
  print("Quantidade de tiles igual")

Total de tiles (count_tiles): 15202
Total de tiles (torch_embed): 15202
Quantidade de tiles igual


## Loading CSV of slide's tiles

In [37]:
tiles_annotation = pd.read_csv(base_dir + "/CSV/" + slide_name + ".csv")
tiles_annotation

,0
0,0_38528_1344.png
1,1_38976_1344.png
2,2_39424_1344.png
3,3_40320_1344.png
4,4_40768_1344.png
...,...
15197,15197_46144_79296.png
15198,15198_46592_79296.png
15199,15199_47040_79296.png
15200,15200_48832_79296.png


## Inference

In [38]:
torch_embeddings.unsqueeze(0).shape

torch.Size([1, 15202, 1024])

In [39]:
with torch.no_grad():
    preds = model(torch_embeddings.unsqueeze(0).to(device))

for i, name in enumerate(["cancer", "candidiasis", "clue cell"]):
    logits = preds[i]
    print(f"{name:<15} {logits}")

cancer          tensor([[-3.3316,  3.0054]])
candidiasis     tensor([[ 0.7500, -1.4304]])
clue cell       tensor([[-2.4431,  2.4750]])
